## 1. Agent simple

In [1]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama
from IPython.display import Markdown, display

MODELE = "qwen3-vl:2b-instruct"
llm = ChatOllama(model=MODELE, temperature=0)


def afficher(reponse):
    display(Markdown(reponse["messages"][-1].content))

In [2]:
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français.",
)

reponse = agent.invoke({"messages": [HumanMessage("Je m'appelle Mohamed.")]})
afficher(reponse)

Salut Mohamed ! Je me nomme aussi. Comment vas-tu ?

In [3]:
# Sans mémoire, la conversation précédente est oubliée
reponse = agent.invoke({"messages": [HumanMessage("Quel est mon nom ?")]})
afficher(reponse)

Je ne sais pas ton nom. Pour me donner un nom, je te remercie d'avance.

### Configuration et contrôle du modèle

In [4]:
llm_configure = ChatOllama(
    model=MODELE,
    temperature=0.1,   # peu de hasard
    num_predict=500,   # nombre maximal de tokens générés
    num_ctx=4096,      # taille du contexte
)

agent = create_agent(model=llm_configure, tools=[], system_prompt="Tu es un assistant concis.")
afficher(agent.invoke({"messages": [HumanMessage("Explique en deux phrases ce qu'est un agent IA.")]}))

Un agent IA est un programme ou un modèle d'intelligence artificielle capable de prendre des décisions ou d'agir selon des règles ou des données. Il peut interagir avec son environnement pour effectuer des tâches comme la prise de décision, la reconnaissance de texte ou la gestion de ressources.

## 2. Mémoire de l'agent

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

agent_memoire = create_agent(
    model=llm,
    tools=[],
    system_prompt="Tu es un assistant utile. Réponds en français.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "conversation-1"}}

agent_memoire.invoke({"messages": [HumanMessage("Je m'appelle Mohamed.")]}, config=config)
afficher(agent_memoire.invoke({"messages": [HumanMessage("Quel est mon nom ?")]}, config=config))

Tu m'appelles Mohamed. Tu veux que je t'aide à trouver quelque chose ?

In [6]:
# Une autre conversation (thread_id différent) ne partage pas la mémoire
autre = {"configurable": {"thread_id": "conversation-2"}}
afficher(agent_memoire.invoke({"messages": [HumanMessage("Quel est mon nom ?")]}, config=autre))

Je ne peux pas savoir ton nom, mais je suis un assistant virtuel. Si tu as besoin d'aide, n'hésite pas à me le dire !

## 3. Tools

In [7]:
from langchain.tools import tool


@tool
def search(query: str) -> dict:
    """Rechercher des actualités."""
    print(f"[tool] search({query})")
    return {"query": query, "news": ["Le temps est très froid", "Les conditions météo sont délicates"]}


@tool
def get_weather(location: str) -> str:
    """Obtenir la météo d'une ville."""
    print(f"[tool] get_weather({location})")
    return f"Météo à {location} : ensoleillé, 32 °C"


@tool
def get_employee_info(name: str) -> dict:
    """Obtenir les informations d'un employé à partir de son nom (salaire, poste)."""
    print(f"[tool] get_employee_info({name})")
    return {"name": name, "salary": 12000, "job": "Développeur"}


agent_tools = create_agent(
    model=llm,
    tools=[search, get_weather, get_employee_info],
    system_prompt="Réponds à la question en utilisant les outils fournis. Réponds en français.",
)

In [8]:
afficher(agent_tools.invoke({"messages": [HumanMessage("Quel temps fait-il à Marrakech ?")]}))

[tool] get_weather(Marrakech)


Le temps à Marrakech est ensoleillé, avec une température de 32 °C.

In [9]:
afficher(agent_tools.invoke({"messages": [HumanMessage("Quel est le salaire et le poste de Mohamed ?")]}))

[tool] get_employee_info(Mohamed)


Le salaire de Mohamed est de 12 000 unités et il est employé comme développeur.

In [10]:
# Voir les étapes ReAct : appel d'outil, observation, réponse
reponse = agent_tools.invoke({"messages": [HumanMessage("Quelles sont les dernières nouvelles ?")]})
for message in reponse["messages"]:
    message.pretty_print()

[tool] search(dernières nouvelles)
================================ Human Message =================================

Quelles sont les dernières nouvelles ?
================================== Ai Message ==================================
Tool Calls:
  search (843feb97-e971-45f5-a64a-dade8800f437)
 Call ID: 843feb97-e971-45f5-a64a-dade8800f437
  Args:
    query: dernières nouvelles
================================= Tool Message =================================
Name: search

{"query": "dernières nouvelles", "news": ["Le temps est très froid", "Les conditions météo sont délicates"]}
================================== Ai Message ==================================

Voici les dernières nouvelles :

1. Le temps est très froid.
2. Les conditions météo sont délicates.


## 4. Tools prédéfinis

### DuckDuckGo

In [11]:
from ddgs import DDGS


@tool
def web_search(query: str, num_results: int = 5) -> str:
    """Rechercher sur le web avec DuckDuckGo.

    Args:
        query: texte de la recherche
        num_results: nombre de résultats (5 par défaut)
    """
    print(f"[tool] web_search({query})")
    try:
        resultats = DDGS().text(query, max_results=num_results)
    except Exception as erreur:
        return f"Recherche impossible : {erreur}"
    if not resultats:
        return f"Aucun résultat pour {query}"
    return "\n".join(f"{i}. {r.get('title')} — {r.get('body')} ({r.get('href')})"
                     for i, r in enumerate(resultats, 1))


agent_web = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="Utilise l'outil web_search pour répondre, puis résume en français avec les liens.",
)
afficher(agent_web.invoke({"messages": [HumanMessage("Cherche ce qu'est LangGraph.")]}))

[tool] web_search(LangGraph qu'est-ce que c'est)


LangGraph est un framework open source conçu pour la création et la gestion de workflows complexes d'agents IA. Il permet de définir, coordonner et exécuter plusieurs agents LLM (langues de programmation) de manière structurée et efficace. Il s'inspire de Pregel et d'Apache Beam, et utilise des architectures basées sur des graphes pour gérer des flux de contrôle complexes. LangGraph est un outil essentiel pour les systèmes d'IA qui nécessitent une orchestration avancée et une gestion des tâches complexes. Il est également utilisé pour créer des agents à état, permettant une réexamen des décisions et une collaboration entre plusieurs agents. Le framework est intégré à LangChain et peut être utilisé sans LangChain. Il est développé par des équipes de recherche et de développement dans le domaine de l'intelligence artificielle.

### Tavily (nécessite `TAVILY_API_KEY` dans le fichier `.env`)

In [12]:
import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(override=True)

if os.getenv("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch

    tavily = TavilySearch(max_results=5)

    @tool
    def tavily_search(query: str) -> str:
        """Rechercher des informations récentes sur le web."""
        print(f"[tool] tavily_search({query})")
        return tavily.invoke({"query": query})

    agent_tavily = create_agent(
        model=llm,
        tools=[tavily_search],
        system_prompt=f"Tu recherches sur le web. Date du jour : {datetime.now():%Y-%m-%d}.",
    )
    afficher(agent_tavily.invoke({"messages": [HumanMessage("Quel temps fait-il aujourd'hui à Casablanca ?")]}))
else:
    print("TAVILY_API_KEY absente : créer une clé gratuite sur tavily.com et l'ajouter au fichier .env")

TAVILY_API_KEY absente : créer une clé gratuite sur tavily.com et l'ajouter au fichier .env


### PythonREPLTool

In [13]:
from langchain_experimental.tools import PythonREPLTool

agent_python = create_agent(
    model=llm,
    tools=[PythonREPLTool()],
    system_prompt="Écris du code Python et exécute-le avec l'outil Python_REPL. Utilise print pour afficher le résultat.",
)
reponse = agent_python.invoke({"messages": [HumanMessage(
    "Trie les listes [6, 5, 8, 3, 2] et [65, 15, 8, 13, 2], puis calcule la somme de chaque liste triée."
)]})
for message in reponse["messages"]:
    message.pretty_print()

C:\Program Files\KMSpico\temp\ipykernel_3968\4104116588.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool
Python REPL can execute arbitrary code. Use with caution.


================================ Human Message =================================

Trie les listes [6, 5, 8, 3, 2] et [65, 15, 8, 13, 2], puis calcule la somme de chaque liste triée.
================================== Ai Message ==================================
Tool Calls:
  Python_REPL (a5c77caa-0078-45ec-ab46-8a0969690377)
 Call ID: a5c77caa-0078-45ec-ab46-8a0969690377
  Args:
    query: def sort_and_sum(list1, list2):
    # Sort both lists
    sorted_list1 = sorted(list1)
    sorted_list2 = sorted(list2)
    # Calculate the sum of each sorted list
    sum1 = sum(sorted_list1)
    sum2 = sum(sorted_list2)
    return sorted_list1, sum1, sorted_list2, sum2

# Test with the given lists
list1 = [6, 5, 8, 3, 2]
list2 = [65, 15, 8, 13, 2]

# Call the function
sorted_list1, sum1, sorted_list2, sum2 = sort_and_sum(list1, list2)

# Print the results
print(f"List 1: {list1}, Sorted: {sorted_list1}, Sum: {sum1}")
print(f"List 2: {list2}, Sorted: {sorted_list2}, Sum: {sum2}")
==================

## 5. Middlewares

### Modèle dynamique (`wrap_model_call`)

In [14]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

modele_test = ChatOllama(model=MODELE, temperature=0.7, num_predict=200)    # rapide et court
modele_prod = ChatOllama(model=MODELE, temperature=0, num_predict=1000)     # précis et complet


@wrap_model_call
def selection_dynamique(request: ModelRequest, handler) -> ModelResponse:
    """Choisir le modèle selon l'environnement fourni dans le contexte."""
    env = (request.runtime.context or {}).get("env", "test")
    modele = modele_prod if env == "prod" else modele_test
    print(f"Environnement : {env} -> modèle {'prod' if env == 'prod' else 'test'}")
    return handler(request.override(model=modele))


agent_dyn = create_agent(model=modele_test, tools=[], middleware=[selection_dynamique])
afficher(agent_dyn.invoke({"messages": [HumanMessage("Donne une définition d'un LLM.")]}, context={"env": "test"}))
afficher(agent_dyn.invoke({"messages": [HumanMessage("Donne une définition d'un LLM.")]}, context={"env": "prod"}))

Environnement : test -> modèle test


Un **langage modulaire de langage** (ou **LLM**, en abrégé) est un modèle de langage évolué qui permet de générer des textes en suivant un style de langage donné par un format de données ou un modèle de langage. Il s'agit d’un modèle de langage à grande échelle qui peut être utilisé pour générer, comprendre ou produire du texte en fonction de données d’entrée. Le terme "langage modulaire de langage" peut être utilisé dans un contexte plus technique ou académique pour désigner un modèle de langage qui peut être construit à partir de modules séparés, chacun traitant un aspect spécifique de la langue, comme le traitement du sens, l’analyse de la structure, ou la génération de textes.

Environnement : prod -> modèle prod


Un **langage modulaire** (ou **LLM**, en anglais **Large Language Model**) est un modèle de langage de grande taille, développé par des chercheurs et des entreprises dans le domaine de l'intelligence artificielle. Il est un modèle d'apprentissage automatique (ML) qui est capable de comprendre, générer et manipuler du texte à grande échelle. Ce type de modèle est entraîné sur des données de texte extraites de sources comme les réseaux sociaux, les publications scientifiques, les livres, les articles, etc.

L'objectif principal d'un LLM est de **générer des textes** qui sont **logiquement cohérents**, **sémantiquement sensibles** et **récupérables**. Il peut répondre à des questions, écrire des textes, rédiger des documents, créer des dialogues, et même produire des contenus comme des nouvelles, des poèmes ou des scripts.

Les modèles LLM sont généralement **très grands** (de plusieurs milliards de paramètres), ce qui leur permet de capturer des **régions de langage** très complexes, comme les nuances du langage, les contextes, les règles de syntaxe, et les structures de phrases. Ils sont souvent **entraînés à partir de données massives** et utilisés dans des applications telles que :

- La génération de contenu (texte, nouvelles, rédaction de documents)
- La prise en charge de conversations (chatbots)
- L'analyse du langage (recherche, traduction, correction de texte)
- La création de contenu personnalisé (suggestions, recommandations)

En résumé, un **LLM** est un modèle d'apprentissage automatique de grande taille capable de comprendre et de produire du texte de manière fluide et intelligente.

### Prompt dynamique (`dynamic_prompt`)

In [15]:
from typing import TypedDict
from langchain.agents.middleware import dynamic_prompt


class Contexte(TypedDict):
    user_role: str


@dynamic_prompt
def prompt_selon_role(request: ModelRequest) -> str:
    role = (request.runtime.context or {}).get("user_role", "user")
    base = "Tu es un assistant utile. Réponds en français."
    if role == "expert":
        return base + " Donne une réponse technique et détaillée."
    if role == "beginner":
        return base + " Explique simplement, sans jargon, en trois phrases."
    return base


agent_role = create_agent(model=llm, tools=[], middleware=[prompt_selon_role], context_schema=Contexte)

for role in ("expert", "beginner"):
    print(f"===== {role} =====")
    afficher(agent_role.invoke({"messages": [HumanMessage("Explique le machine learning.")]},
                               context={"user_role": role}))

===== expert =====


Le **machine learning** est une branche de l'intelligence artificielle qui consiste à permettre aux ordinateurs d'apprendre et de se développer à partir de données, en se basant sur l'analyse de ces données pour améliorer leurs performances sur des tâches spécifiques. Il s'agit d'une méthode d'apprentissage automatique qui ne repose pas sur des règles préprogrammées, mais sur l'observation et l'analyse de données pour apprendre à partir de données d'entrées et de sorties.

### 1. **Définition simple**
Le machine learning consiste à entraîner un modèle d'ordinateur à effectuer des prédictions ou des classifications en utilisant des données historiques. L'ordinateur apprend à partir de ces données en se basant sur des algorithmes qui permettent d'identifier des patterns ou des corrélations entre les données d'entrées et les données de sortie.

### 2. **Types de machine learning**
Il existe plusieurs types de machine learning, selon la manière dont les données sont utilisées :

- **Machine learning supervisée (ou apprentissage supervisé)** : Le modèle est entraîné à partir de données avec des étiquettes (ex : classification, régression). Exemple : un modèle qui apprend à classer un image comme « animal » ou « personne ».
  
- **Machine learning non supervisée (ou apprentissage non supervisé)** : Le modèle ne reçoit pas d'étiquettes. Il cherche à détecter des groupes ou des structures dans les données. Exemple : clustering (groupes de données similaires).

- **Machine learning hybride** : Combinaison de plusieurs types de machine learning.

- **Machine learning à grande échelle** : Utilisation de grandes quantités de données pour améliorer les performances.

### 3. **Processus de machine learning**
Le processus de machine learning comprend plusieurs étapes :

1. **Données d'entrées** : Données qui sont utilisées pour entraîner le modèle.
2. **Choix du modèle** : Choisir un modèle approprié selon le type de problème.
3. **Entraînement du modèle** : L'ordinateur apprend à partir des données en utilisant des algorithmes d'apprentissage.
4. **Validation et test** : Évaluer la performance du modèle sur des données non utilisées.
5. **Application du modèle** : Utiliser le modèle pour effectuer des prédictions ou des classifications.

### 4. **Exemples de machine learning**
- **Classification** : Exemple : détecter si une image est une photo de voiture ou non.
- **Régression** : Exemple : prédire le prix d'une maison en fonction de ses caractéristiques.
- **Reconnaissance d'images** : Exemple : détecter des objets dans une image.
- **Recommandation** : Exemple : recommander des films à regarder en fonction des préférences des utilisateurs.

### 5. **Applications réelles**
Le machine learning est utilisé dans de nombreux domaines :
- **Santé** : Diagnostic de maladies à partir de données médicales.
- **Finance** : Analyse de marché, prédiction des prix.
- **Marketing** : Réalisation de recommandations personnalisées.
- **Réseaux sociaux** : Suivi des comportements des utilisateurs.
- **Automatisation** : Automatisation de tâches complexes.

### 6. **Limites et défis**
- La qualité des données est cruciale.
- Le modèle peut être sujet à des erreurs si les données sont mal choisies ou mal traitées.
- L'interprétation des résultats peut être difficile.

---

En résumé, le machine learning est une technologie qui permet aux ordinateurs d'apprendre à partir de données et de s'adapter à des tâches complexes, en utilisant des algorithmes d'apprentissage automatique. Il est essentiel dans l'intelligence artificielle moderne et est utilisé dans de nombreux domaines de l'industrie, de la santé, et de la recherche.

===== beginner =====


Le machine learning est une technique qui permet aux ordinateurs d'apprendre à partir de données, comme les exemples de données que l'on leur donne. Il n'est pas un programme qui fait des calculs, mais qui apprend à partir de données pour mieux faire des prédictions ou des décisions. Par exemple, il peut apprendre à reconnaître des visages ou à classer des images.

### Gestion des erreurs des tools (`wrap_tool_call`)

In [16]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage


@tool
def diviser(a: float, b: float) -> float:
    """Diviser a par b."""
    print(f"[tool] diviser({a}, {b})")
    return a / b


@wrap_tool_call
def gerer_erreurs_tools(request, handler):
    """Transformer une exception d'un tool en message compréhensible pour le modèle."""
    try:
        return handler(request)
    except Exception as erreur:
        print(f"Erreur interceptée : {erreur!r}")
        return ToolMessage(
            content=f"Erreur de l'outil : vérifie les paramètres ({erreur}).",
            tool_call_id=request.tool_call["id"],
        )


agent_erreurs = create_agent(model=llm, tools=[diviser], middleware=[gerer_erreurs_tools],
                             system_prompt="Utilise l'outil diviser pour les divisions.")
reponse = agent_erreurs.invoke({"messages": [HumanMessage("Combien font 10 divisé par 0 ?")]})
for message in reponse["messages"]:
    message.pretty_print()

================================ Human Message =================================

Combien font 10 divisé par 0 ?
================================== Ai Message ==================================

La division de 10 par 0 est indéfinie, car il n'est pas possible de diviser par zéro. 

Utilisant l'outil `diviser`, je ne peux pas effectuer cette division car il s'agit d'une division par zéro, ce qui est invalide.

Donc, la réponse est : **l'opération est impossible**.


### Middleware personnalisé (hooks)

In [17]:
import time
from langchain.agents.middleware import AgentMiddleware, AgentState


class TraceHooks(AgentMiddleware):
    def before_agent(self, state: AgentState, runtime):
        self.debut = time.time()
        print("before_agent")

    def before_model(self, state: AgentState, runtime):
        print("before_model")

    def after_model(self, state: AgentState, runtime):
        print("after_model")

    def after_agent(self, state: AgentState, runtime):
        print(f"after_agent — durée : {time.time() - self.debut:.1f} s")


agent_hooks = create_agent(model=llm, tools=[get_weather], middleware=[TraceHooks()])
afficher(agent_hooks.invoke({"messages": [HumanMessage("Météo à Rabat ?")]}))

before_agent
before_model
after_model
[tool] get_weather(Rabat)
before_model
after_model
after_agent — durée : 8.7 s


La météo à Rabat est ensoleillée avec une température de 32 °C.

### Sortie structurée

In [18]:
from pydantic import BaseModel
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str


agent_contact = create_agent(model=llm, tools=[], response_format=ToolStrategy(ContactInfo))
resultat = agent_contact.invoke({"messages": [HumanMessage(
    "Extrais les coordonnées : Mohamed YOUSSFI, med@gmail.com, (212) 123-4567")]})
print(resultat["structured_response"])

name='Mohamed YOUSSFI' email='med@gmail.com' phone='(212) 123-4567'


## 6. Guardrails

### Détection des données personnelles (PII)

In [19]:
from langchain.agents.middleware import PIIMiddleware


@tool
def send_email(email_address: str) -> str:
    """Envoyer un email à l'adresse donnée."""
    print(f"[tool] send_email({email_address})")
    return f"Email envoyé à {email_address}"


@tool
def card_tool(credit_card: str) -> str:
    """Effectuer un paiement avec une carte bancaire."""
    print(f"[tool] card_tool({credit_card})")
    return f"Paiement effectué avec {credit_card}"


agent_pii = create_agent(
    model=llm,
    tools=[send_email, card_tool],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("api_key", detector=r"sk-[a-zA-Z0-9]{32}", strategy="block", apply_to_input=True),
    ],
)

reponse = agent_pii.invoke({"messages": [HumanMessage(
    "Envoie un email à med@gmail.com et paie avec la carte 5105-1051-0510-5100.")]})
for message in reponse["messages"]:
    message.pretty_print()

[tool] send_email([REDACTED_EMAIL])
[tool] card_tool(****-****-****-5100)
================================ Human Message =================================

Envoie un email à [REDACTED_EMAIL] et paie avec la carte ****-****-****-5100.
================================== Ai Message ==================================
Tool Calls:
  send_email (8f0a2ca3-931c-43e6-8ac0-9cdd5db4194e)
 Call ID: 8f0a2ca3-931c-43e6-8ac0-9cdd5db4194e
  Args:
    email_address: [REDACTED_EMAIL]
  card_tool (996559d5-fd02-4b8e-9dac-6ef888f7cf5a)
 Call ID: 996559d5-fd02-4b8e-9dac-6ef888f7cf5a
  Args:
    credit_card: ****-****-****-5100
================================= Tool Message =================================
Name: send_email

Email envoyé à [REDACTED_EMAIL]
================================= Tool Message =================================
Name: card_tool

Paiement effectué avec ****-****-****-5100
================================== Ai Message ==================================

L'email a été envoyé à [REDACTED_

In [20]:
# Une clé API dans la demande bloque l'exécution
try:
    agent_pii.invoke({"messages": [HumanMessage("Utilise la clé sk-" + "a" * 32)]})
except Exception as erreur:
    print("Bloqué :", type(erreur).__name__, erreur)

Bloqué : PIIDetectionError Detected 1 instance(s) of api_key in text content


### Guardrail déterministe avant l'agent (`before_agent`)

In [21]:
from typing import Any
from langchain.agents.middleware import before_agent, after_agent
from langgraph.runtime import Runtime

mots_interdits = ["hack", "exploit", "malware"]


@before_agent(can_jump_to=["end"])
def filtre_contenu(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Bloquer les demandes contenant des mots interdits."""
    premier = state["messages"][0]
    if premier.type != "human":
        return None
    texte = str(premier.content).lower()
    if any(mot in texte for mot in mots_interdits):
        return {"messages": [{"role": "assistant",
                              "content": "Je ne peux pas traiter cette demande. Merci de la reformuler."}],
                "jump_to": "end"}
    return None


agent_filtre = create_agent(model=llm, tools=[], middleware=[filtre_contenu])
afficher(agent_filtre.invoke({"messages": [HumanMessage("Comment hack une base de données ?")]}))

Je ne peux pas traiter cette demande. Merci de la reformuler.

### Guardrail basé sur un modèle après l'agent (`after_agent`)

In [22]:
from langchain.messages import AIMessage

modele_securite = ChatOllama(model=MODELE, temperature=0)


@after_agent(can_jump_to=["end"])
def verification_securite(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Faire évaluer la réponse finale par un modèle."""
    dernier = state["messages"][-1]
    if not isinstance(dernier, AIMessage):
        return None
    verdict = modele_securite.invoke(
        "Cette réponse est-elle sûre et appropriée ? Réponds uniquement SAFE ou UNSAFE.\n"
        f"Réponse : {dernier.content}"
    ).content
    print("Verdict du modèle de sécurité :", verdict)
    if "UNSAFE" in verdict.upper():
        dernier.content = "Je ne peux pas fournir cette réponse."
    return None


agent_securise = create_agent(model=llm, tools=[], middleware=[verification_securite])
afficher(agent_securise.invoke({"messages": [HumanMessage("Donne-moi une recette de crêpes.")]}))

Verdict du modèle de sécurité : SAFE


Bien sûr ! Voici une recette de crêpes simple et délicieuse que vous pouvez faire maison. C'est une version légère et savoureuse, idéale pour un petit déjeuner ou un goûter.

---

### 🍳 **Crêpes aux fruits et miel (facile et délicieuse)**

**Ingrédients :**

- 4 œufs
- 150 g de farine (ou 150 g de pâte à crêpes)
- 100 g de sucre
- 100 g de miel
- 100 g de fruits frais (comme fraises, framboises, ou poires)
- 1 cuillère à café de vanille
- 1 cuillère à café de sel
- 1 cuillère à café de poudre de levure (ou 1 cuillère à café de bicarbonate de soude)
- 1 cuillère à café de lait (ou 100 ml d’eau)
- 1 cuillère à café de beurre (ou 100 g de beurre)
- 1 cuillère à café de cacao (optionnel)

---

### 🍳 **Instructions :**

1. **Préparation de la pâte :**
   - Dans un grand bol, mélangez la farine, le sucre, le sel, la levure, le lait, et l’œuf.
   - Ajoutez le miel et mélangez bien.
   - Si vous préférez, ajoutez un peu de beurre et mélangez jusqu’à une consistance lisse.

2. **Faire les crêpes :**
   - Préchauffez votre gril à 180°C.
   - Dans une poêle antiadhésive, faites chauffer l’huile.
   - Versez une petite quantité de pâte dans la poêle, et faites une petite boule de crêpe.
   - Laissez cuire 2 à 3 minutes, jusqu’à ce que la crêpe soit dorée.
   - Retirez-la de la poêle et laissez-la refroidir.

3. **Décoration :**
   - Lorsque les crêpes sont cuites, déposez-les sur une assiette.
   - Ajoutez les fruits frais et le miel.
   - Si vous préférez, ajoutez un peu de cacao ou de vanille.

---

### 🍽️ **Astuce :**
- Pour une crêpe plus moelleuse, ajoutez un peu de lait de coco ou de lait de vache.
- Pour une crêpe plus légère, utilisez de la farine de riz ou de blé.

---

Souhaitez-vous une version plus sucrée, plus salée, ou avec des ingrédients spécifiques comme des fruits ou des légumes ? Je peux vous proposer une recette adaptée ! 😊

## 7. Human In The Loop

In [23]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

agent_hitl = create_agent(
    model=llm,
    tools=[send_email],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "hitl-1"}}
resultat = agent_hitl.invoke({"messages": [HumanMessage("Envoie un email à med@gmail.com")]}, config=config)
print(resultat["__interrupt__"])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'email_address': 'med@gmail.com'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'email_address': 'med@gmail.com'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='b9d3ebe2cca26cdb3bc2ced809ad8a0f', response_schema=None)]


In [24]:
# L'humain approuve : l'outil est exécuté
resultat = agent_hitl.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)
print(resultat["messages"][-1].content)

[tool] send_email(med@gmail.com)
L'email a été envoyé à med@gmail.com.


In [25]:
# Nouvelle demande, cette fois refusée par l'humain : l'outil n'est pas exécuté
config = {"configurable": {"thread_id": "hitl-2"}}
agent_hitl.invoke({"messages": [HumanMessage("Envoie un email à test@example.com")]}, config=config)
resultat = agent_hitl.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Envoi refusé par l'utilisateur."}]}),
    config=config,
)
print(resultat["messages"][-1].content)

L'outil de transmission d'email a été rejeté par l'utilisateur. Veuillez vérifier les détails de l'adresse email et assurez-vous que l'adresse est valide et accessible. Si le problème persiste, veuillez me fournir plus d'informations pour mieux vous aider.


## 8. Chatbot basé sur un RAG agentique

Le RAG devient un **tool** : l'agent décide quand chercher dans le PDF et peut lancer plusieurs recherches. La mémoire (`thread_id`) permet de poser des questions de suivi.

In [26]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

documents = PyPDFLoader("pdfs/attention_is_all_you_need.pdf").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print(len(documents), "pages |", len(chunks), "chunks")

C:\Program Files\KMSpico\temp\ipykernel_3968\3063055527.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


11 pages | 51 chunks


In [27]:
@tool
def retriever_tool(query: str) -> str:
    """Rechercher des passages dans l'article « Attention Is All You Need » :
    architecture Transformer, attention, multi-head attention, encodage positionnel,
    entraînement, résultats (BLEU) et hyperparamètres."""
    print(f"[tool] retriever_tool({query})")
    documents = retriever.invoke(query)
    return "\n\n".join(f"[page {d.metadata.get('page', 0) + 1}] {d.page_content}" for d in documents)


chatbot = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=(
        "Tu es un assistant qui répond aux questions sur l'article « Attention Is All You Need ». "
        "Utilise retriever_tool pour chaque information du document ; pour une question composée, "
        "fais une recherche par sous-question. Cite les pages. "
        "Si l'information n'est pas dans les passages, réponds « Je ne sais pas »."
    ),
    checkpointer=InMemorySaver(),
)


def discuter(question, thread_id="chat-rag"):
    reponse = chatbot.invoke({"messages": [HumanMessage(question)]},
                             config={"configurable": {"thread_id": thread_id}})
    appels = [a["args"] for m in reponse["messages"] if isinstance(m, AIMessage) for a in m.tool_calls]
    print("Recherches lancées par l'agent :", appels[-3:])
    display(Markdown(f"**Vous :** {question}\n\n**Assistant :** {reponse['messages'][-1].content}"))

In [28]:
discuter("Quels sont les auteurs et la contribution principale de l'article ?")

Recherches lancées par l'agent : []


**Vous :** Quels sont les auteurs et la contribution principale de l'article ?

**Assistant :** Je ne sais pas.

In [29]:
discuter("Et combien de couches et de têtes d'attention utilise le modèle de base ?")

[tool] retriever_tool(architecture Transformer, attention, multi-head attention, encodage positionnel, entraînement, résultats (BLEU) et hyperparamètres)
Recherches lancées par l'agent : [{'query': 'architecture Transformer, attention, multi-head attention, encodage positionnel, entraînement, résultats (BLEU) et hyperparamètres'}]


**Vous :** Et combien de couches et de têtes d'attention utilise le modèle de base ?

**Assistant :** L'article mention que le modèle de base utilise **8 couches** et **8 têtes d'attention**. 

Plus précisément :
- Le modèle utilise **8 couches** (ou **8 couches d'attention**) avec **8 têtes d'attention**.
- Chaque tête d'attention utilise une dimension de 64, et les projections sont paramétrées avec des matrices $ W_Q, W_K, W_V $ et $ W_O $.

Ces informations sont tirées des pages 5 et 9 de l'article.

In [30]:
discuter("Compare le rôle de l'attention multi-têtes et de l'encodage positionnel.")

Recherches lancées par l'agent : [{'query': 'architecture Transformer, attention, multi-head attention, encodage positionnel, entraînement, résultats (BLEU) et hyperparamètres'}]


**Vous :** Compare le rôle de l'attention multi-têtes et de l'encodage positionnel.

**Assistant :** Le rôle de l'attention multi-têtes et de l'encodage positionnel est décrit dans l'article comme suit :

- **L'attention multi-têtes** est utilisée pour permettre à chaque couche de l'attention de traiter des informations différentes. Elle permet de mieux capturer les relations entre les mots dans un texte, en utilisant plusieurs têtes d'attention pour analyser les données de manière plus complète. L'attention multi-têtes permet de mieux capturer les relations entre les mots dans un texte, en utilisant plusieurs têtes d'attention pour analyser les données de manière plus complète.

- **L'encodage positionnel** est utilisé pour permettre à chaque couche de l'attention de traiter les informations de manière plus complète. Il permet de mieux capturer les relations entre les mots dans un texte, en utilisant plusieurs têtes d'attention pour analyser les données de manière plus complète.

Ces informations sont tirées des pages 9 et 5 de l'article.

In [31]:
discuter("Quelle accuracy GPT-4 obtient-il sur MMLU selon cet article ?")

Recherches lancées par l'agent : [{'query': 'architecture Transformer, attention, multi-head attention, encodage positionnel, entraînement, résultats (BLEU) et hyperparamètres'}]


**Vous :** Quelle accuracy GPT-4 obtient-il sur MMLU selon cet article ?

**Assistant :** Je ne sais pas.